In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga del shapefile de secciones censales

In [2]:
# Cargo el GDF con las secciones censales
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)

# Compruebo que se haya cargado
print(f"Vista del gdf de secciones:\n{gdf_secciones.head()}")

Vista del gdf de secciones:
        CUSEC  CUMUN CSEC CDIS CMUN CPRO CCA    CUDIS  CLAU2   NPRO  \
0  0500101001  05001  001   01  001   05  07  0500101  05001  Ávila   
1  0500201001  05002  001   01  002   05  07  0500201  05002  Ávila   
2  0500201002  05002  002   01  002   05  07  0500201  05002  Ávila   
3  0500501001  05005  001   01  005   05  07  0500501  05005  Ávila   
4  0500701001  05007  001   01  007   05  07  0500701  05007  Ávila   

               NCA CNUT0 CNUT1 CNUT2 CNUT3                      NMUN  \
0  Castilla y León    ES     4     1     1                   Adanero   
1  Castilla y León    ES     4     1     1                Adrada, La   
2  Castilla y León    ES     4     1     1                Adrada, La   
3  Castilla y León    ES     4     1     1                  Albornos   
4  Castilla y León    ES     4     1     1  Aldeanueva de Santa Cruz   

                                            geometry  
0  POLYGON ((365705.918 4536187.034, 365958.915 4...  
1 

# Carga del fichero de CEAS

Como no hay publicado (o no se ha localizado) ningún fichero csv con la información de los CEAS, se ha inspeccionado
el tráfico de red de la web CATDISS en su mapa de recursos filtrando por los CEAS, y con la librearía requests se ha
replicado la llamada a la API que se realiza a nivel provincial para obtener un csv con los datos de Castilla y León.
Esta obtención se ha tratado en un cuaderno a parte y se ha guardado la información en un CSV.

https://servicios.jcyl.es/catdiss/#/mapa

In [3]:
# Cargo el fichero csv con los centros docentes
ruta_csv_ceas = os.path.join(DATA_INPUTS_DA, "ceas.csv")
df_ceas = pd.read_csv(ruta_csv_ceas, sep=",", encoding="utf-8")

# Asigno un ID para despues relacionar
df_ceas["ceas_id"] = range(1, len(df_ceas) + 1)

# Veo una muestra de su estructura y contenido
print(df_ceas.info())
df_ceas.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Nombre     215 non-null    object 
 1   Latitud    211 non-null    float64
 2   Longitud   211 non-null    float64
 3   Provincia  215 non-null    object 
 4   ceas_id    215 non-null    int64  
dtypes: float64(2), int64(1), object(2)
memory usage: 8.5+ KB
None


,Nombre,Latitud,Longitud,Provincia,ceas_id
53,CEAS SANTA MARÍA DEL CAMPO,42.132088,-3.975429,Burgos,54
4,CEAS ARAVALLE - BARCO DE ÁVILA,40.359087,-5.519202,Ávila,5
111,CEAS CARRIÓN DE LOS CONDES - OSORNO (SEDE II),42.411643,-4.361823,Palencia,112
209,CEAS ZAMORA RURAL,41.510184,-5.731469,Zamora,210
152,CEAS PINARES SUR,41.829463,-3.067906,Soria,153


# Asignación de código de sección censal (CUSEC)
Como no se dispone por defecto del CUSEC pero sí de las coordenadas geográficas del CEAS, y se dispone de un shapefile con los
códigos de sección de Castilla y León georreferenciados, puede asignarse el CUSEC a este conjunto de datos mediante un join espacial

In [4]:
# Utilizo una función definida para integrar el cusec
df_ceas_final = asignar_cusec_por_coordenadas(
    df_ceas, "Longitud", "Latitud")

# Compruebo que tiene ahora la columna cusec
df_ceas_final.sample(5)

,Nombre,Latitud,Longitud,Provincia,ceas_id,CUSEC
127,CEAS ROLLO,40.965020,-5.650455,Salamanca,128,3727402011
59,CEAS LEÓN II -VIRGEN DEL CAMINO,42.580464,-5.642792,León,60,2418901002
6,CEAS MORAÑA ALTA - MADRIGAL DE LAS ALTAS TORRE...,41.089367,-4.998595,Ávila,7,0511402001
81,CEAS PUENTE CASTRO - SAN CLAUDIO,42.589427,-5.564595,León,82,2408906006
136,CEAS CANTALEJO - FUENTESAÚCO DE FUENTIDUEÑA,41.255667,-3.928296,Segovia,137,4004001001


In [5]:
# Miro a ver si algun CEAS queda sin CUSEC
print(f"CEAS sin CUSEC asignado: {df_ceas_final['CUSEC'].isna().sum()}")

print(f"Detalle:\n{df_ceas_final[df_ceas_final['CUSEC'].isna()]['Nombre']}")

# Como son 4, puedo consultar el CP manualmente los códigos postales de esas localidades, y despues hacer la consulta por CP
mapa_cp = {
    "CEAS VALLE DEL CORNEJA - PIEDRAHITA (SEDE I)": "05500",
    "CEAS SALDAÑA": "34100",
    "CEAS VENTA DE BAÑOS - DUEÑAS": "34210",
    "CEAS ZAMORA SUR": "49011",
}

# Creo la columna 'CP' según la columna 'Nombre'
df_ceas_final["CP"] = df_ceas_final["Nombre"].map(mapa_cp)

# Y ahora ya consulto obtengo el CUSEC por CP
df_ceas_final = asignar_cusec_por_cp(df_ceas_final)

# Elimino la columna CP ya que no me sirve
df_ceas_final.drop(columns=["CP"], inplace=True)

CEAS sin CUSEC asignado: 4
Detalle:
16     CEAS VALLE DEL CORNEJA - PIEDRAHITA (SEDE I)
112                                    CEAS SALDAÑA
113                    CEAS VENTA DE BAÑOS - DUEÑAS
212                                 CEAS ZAMORA SUR
Name: Nombre, dtype: object
✅ 215 registros procesados | 🔁 4 CUSEC completados | ❗ 0 sin asignar


# Cálculo de distancia mínima a un CEAS desde un CUSEC y accesibilidad de CEAS para un CUSEC
Es necesario para cada sección censal calcular la distancia mínima existente hacia un CEAS , así como crear
una medida de accesibilidad desde una sección censal

In [6]:
df_resultado, df_relaciones = calcular_accesibilidad_v2(
    df_servicio=df_ceas_final,
    nombre_servicio="ceas",
    col_id = "ceas_id",
)

df_resultado.head()

Accesibilidad ceas: 100%|████████████████████████████████████████████████████| 3535/3535 [00:09<00:00, 386.82sección/s]


,CUSEC,dist_min_ceas_km,n_ceas_1km,n_ceas_5km,n_ceas_15km,n_ceas_30km,disp_ponderada_ceas
0,0500101001,5.688878,0,0,1,3,0.5
1,0500201001,6.277992,0,0,3,6,1.2
2,0500201002,0.000000,1,1,4,6,2.1
3,0500501001,7.867082,0,0,1,8,1.0
4,0500701001,7.357844,0,0,2,5,0.9


In [7]:
df_relaciones.head()

,CUSEC_origen,ceas_id,CUSEC_servicio,distancia
0,0500101001,19,0520401001,5688.878135
1,0500101001,6,0501601001,18790.534947
2,0500101001,1,0501903002,28508.955243
4,0500201001,9,0500201002,6593.319476
5,0500201001,21,0505401001,9715.216842


# Export de los csv construidos

In [10]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DA, exist_ok=True)

# Rutas de salida
ruta_resultado = os.path.join(DATA_OUTPUTS_DA, "accesibilidad_ceas.csv")
ruta_ceas = os.path.join(DATA_OUTPUTS_DA, "ceas_final.csv")
ruta_relaciones = os.path.join(DATA_OUTPUTS_DA, "relaciones_ceas.csv")

# Guardar DataFrames
df_resultado.to_csv(ruta_resultado, index=False, encoding="utf-8-sig")

df_ceas_final.to_csv(
    ruta_ceas,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
df_relaciones.to_csv(
    ruta_relaciones,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en:\n- {DATA_OUTPUTS_DA}")

✅ Archivos guardados correctamente en:
- D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DA_Dim_servicios
